# Tutorial E: Mathematical Objects as Classes

**Implementing mathematical structures using object-oriented programming**

---

## References and Further Resources

### Key References
- Strang, G. (2016). *Introduction to Linear Algebra* (5th ed.). Wellesley-Cambridge Press.
- Cormen, T. H., Leiserson, C. E., Rivest, R. L., & Stein, C. (2009). *Introduction to Algorithms* (3rd ed.). MIT Press. Chapter 28: Matrix Operations.
- Stewart, J. (2015). *Calculus: Early Transcendentals* (8th ed.). Cengage Learning.

### Further Exploration
- SymPy Documentation: Symbolic mathematics in Python - [docs.sympy.org](https://docs.sympy.org/)
- NumPy's polynomial module: [numpy.org/doc/stable/reference/routines.polynomials.html](https://numpy.org/doc/stable/reference/routines.polynomials.html)
- 3Blue1Brown: "Essence of Linear Algebra" video series

---

## Table of Contents

1. [Introduction: Mathematics Meets Code](#introduction)
2. [Polynomials: Algebraic Expressions as Objects](#polynomials)
3. [Vectors: Geometry and Linear Algebra](#vectors)
4. [Matrices: Linear Transformations](#matrices)
5. [Composition: Building Complex Mathematical Systems](#composition)
6. [Applications in Machine Learning](#ml-applications)
7. [Summary and Key Principles](#summary)

---

## 1. Introduction: Mathematics Meets Code <a id='introduction'></a>

Mathematical objects like polynomials, vectors, and matrices have:
- **State**: Coefficients, components, or entries
- **Operations**: Addition, multiplication, composition
- **Properties**: Degree, magnitude, determinant

Object-oriented programming provides a natural way to represent these structures. Let's explore how we can encode mathematical abstractions as Python classes that are both mathematically rigorous and computationally efficient.

### Why This Matters

In machine learning and scientific computing:
- Neural networks use matrix operations for transformations
- Optimization algorithms work with vectors and gradients
- Feature engineering often involves polynomial transformations
- Understanding these fundamentals helps us build and debug complex systems

Let's build these mathematical objects from scratch.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Union, Tuple
from math import sqrt, acos, pi

# Set up plotting style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)

---

## 2. Polynomials: Algebraic Expressions as Objects <a id='polynomials'></a>

A polynomial is an expression of the form:
$$p(x) = a_n x^n + a_{n-1} x^{n-1} + \cdots + a_1 x + a_0$$

Let's create a class that represents polynomials and supports mathematical operations.

In [ ]:
class Polynomial:
    """A polynomial with real coefficients.
    
    Coefficients are stored in ascending order of powers:
    [a0, a1, a2, ...] represents a0 + a1*x + a2*x^2 + ...
    """
    
    def __init__(self, coefficients: List[float]) -> None:
        """
        Initialize a polynomial with given coefficients.
        
        Args:
            coefficients: List of coefficients [a0, a1, a2, ...]
        """
        # Remove trailing zeros to maintain canonical form
        while len(coefficients) > 1 and coefficients[-1] == 0:
            coefficients.pop()
        
        self.coefficients = coefficients if coefficients else [0]
    
    @property
    def degree(self) -> int:
        """Return the degree of the polynomial."""
        return len(self.coefficients) - 1
    
    def evaluate(self, x_value: float) -> float:
        """
        Evaluate the polynomial at a given point using Horner's method.
        
        Horner's method is numerically stable and efficient:
        p(x) = a0 + x(a1 + x(a2 + x(...)))
        
        Args:
            x_value: Point at which to evaluate
            
        Returns:
            The value p(x)
        """
        result = 0
        for coefficient in reversed(self.coefficients):
            result = result * x_value + coefficient
        return result
    
    def __call__(self, x_value: float) -> float:
        """Allow polynomial to be called like a function."""
        return self.evaluate(x_value)
    
    def __add__(self, other: 'Polynomial') -> 'Polynomial':
        """
        Add two polynomials.
        
        Returns a new polynomial (immutable operation).
        """
        if not isinstance(other, Polynomial):
            return NotImplemented
        
        # Pad shorter coefficient list with zeros
        max_len = max(len(self.coefficients), len(other.coefficients))
        coeffs1 = self.coefficients + [0] * (max_len - len(self.coefficients))
        coeffs2 = other.coefficients + [0] * (max_len - len(other.coefficients))
        
        # Add corresponding coefficients
        new_coeffs = [c1 + c2 for c1, c2 in zip(coeffs1, coeffs2)]
        
        return Polynomial(new_coeffs)
    
    def __sub__(self, other: 'Polynomial') -> 'Polynomial':
        """Subtract two polynomials."""
        if not isinstance(other, Polynomial):
            return NotImplemented
        
        max_len = max(len(self.coefficients), len(other.coefficients))
        coeffs1 = self.coefficients + [0] * (max_len - len(self.coefficients))
        coeffs2 = other.coefficients + [0] * (max_len - len(other.coefficients))
        
        new_coeffs = [c1 - c2 for c1, c2 in zip(coeffs1, coeffs2)]
        
        return Polynomial(new_coeffs)
    
    def __mul__(self, other: Union['Polynomial', float]) -> 'Polynomial':
        """
        Multiply polynomial by another polynomial or scalar.
        
        For polynomial multiplication, we use convolution:
        (a0 + a1*x)(b0 + b1*x) = a0*b0 + (a0*b1 + a1*b0)*x + a1*b1*x^2
        """
        if isinstance(other, Polynomial):
            # Polynomial multiplication
            degree = self.degree + other.degree
            new_coeffs = [0] * (degree + 1)
            
            for i, coeff1 in enumerate(self.coefficients):
                for j, coeff2 in enumerate(other.coefficients):
                    new_coeffs[i + j] += coeff1 * coeff2
            
            return Polynomial(new_coeffs)
        
        elif isinstance(other, (int, float)):
            # Scalar multiplication
            new_coeffs = [c * other for c in self.coefficients]
            return Polynomial(new_coeffs)
        
        return NotImplemented
    
    def __rmul__(self, other: float) -> 'Polynomial':
        """Support scalar * polynomial."""
        return self.__mul__(other)
    
    def derivative(self) -> 'Polynomial':
        """
        Compute the derivative of the polynomial.
        
        Using the power rule: d/dx(a*x^n) = n*a*x^(n-1)
        
        Returns:
            A new polynomial representing the derivative
        """
        if self.degree == 0:
            return Polynomial([0])
        
        new_coeffs = [
            i * self.coefficients[i] 
            for i in range(1, len(self.coefficients))
        ]
        
        return Polynomial(new_coeffs)
    
    def integral(self, constant: float = 0) -> 'Polynomial':
        """
        Compute the indefinite integral of the polynomial.
        
        Using the power rule: ∫(a*x^n)dx = a*x^(n+1)/(n+1) + C
        
        Args:
            constant: The constant of integration
            
        Returns:
            A new polynomial representing the integral
        """
        new_coeffs = [constant] + [
            self.coefficients[i] / (i + 1)
            for i in range(len(self.coefficients))
        ]
        
        return Polynomial(new_coeffs)
    
    def definite_integral(self, lower_bound: float, upper_bound: float) -> float:
        """
        Compute the definite integral over an interval.
        
        Uses the fundamental theorem of calculus:
        ∫[a,b] f(x)dx = F(b) - F(a) where F is an antiderivative of f
        
        Args:
            lower_bound: Lower limit of integration
            upper_bound: Upper limit of integration
            
        Returns:
            The value of the definite integral
        """
        antiderivative = self.integral()
        return antiderivative(upper_bound) - antiderivative(lower_bound)
    
    def __repr__(self) -> str:
        """Create a readable string representation."""
        if self.degree == 0:
            return f"{self.coefficients[0]}"
        
        terms = []
        for i, coeff in enumerate(self.coefficients):
            if coeff == 0:
                continue
            
            # Format coefficient
            if i == 0:
                terms.append(f"{coeff}")
            elif i == 1:
                if coeff == 1:
                    terms.append("x")
                elif coeff == -1:
                    terms.append("-x")
                else:
                    terms.append(f"{coeff}*x")
            else:
                if coeff == 1:
                    terms.append(f"x^{i}")
                elif coeff == -1:
                    terms.append(f"-x^{i}")
                else:
                    terms.append(f"{coeff}*x^{i}")
        
        return ' + '.join(terms).replace('+ -', '- ')
    
    @classmethod
    def from_roots(cls, roots: List[float]) -> 'Polynomial':
        """
        Create a polynomial from its roots.
        
        If r1, r2, ... are roots, then:
        p(x) = (x - r1)(x - r2)...
        
        Args:
            roots: List of polynomial roots
            
        Returns:
            A polynomial with the given roots
        """
        # Start with p(x) = 1
        result = cls([1])
        
        # Multiply by (x - root) for each root
        for root in roots:
            factor = cls([-root, 1])  # (x - root)
            result = result * factor
        
        return result

### Exploring Polynomial Operations

Let's see our polynomial class in action:

In [ ]:
# Create polynomials: p(x) = 1 + 2x + x^2 and q(x) = 1 - x
poly_p = Polynomial([1, 2, 1])  # 1 + 2x + x^2
poly_q = Polynomial([1, -1])     # 1 - x

print(f"p(x) = {poly_p}")
print(f"q(x) = {poly_q}")
print(f"\nDegree of p(x): {poly_p.degree}")
print(f"p(2) = {poly_p(2)}")

# Arithmetic operations
print(f"\np(x) + q(x) = {poly_p + poly_q}")
print(f"p(x) - q(x) = {poly_p - poly_q}")
print(f"p(x) * q(x) = {poly_p * poly_q}")
print(f"3 * p(x) = {3 * poly_p}")

# Calculus operations
print(f"\np'(x) = {poly_p.derivative()}")
print(f"∫p(x)dx = {poly_p.integral()}")
print(f"∫[0,1] p(x)dx = {poly_p.definite_integral(0, 1):.4f}")

### Visualizing Polynomials and Their Derivatives

In [ ]:
def visualize_polynomial_calculus() -> None:
    """Visualize a polynomial and its derivatives."""
    
    # Create a polynomial with interesting behavior
    poly = Polynomial([-1, 0, 1])  # f(x) = x^2 - 1
    first_deriv = poly.derivative()   # f'(x) = 2x
    second_deriv = first_deriv.derivative()  # f''(x) = 2
    
    # Generate points
    x_values = np.linspace(-2, 2, 200)
    y_values = [poly(x) for x in x_values]
    y_first = [first_deriv(x) for x in x_values]
    y_second = [second_deriv(x) for x in x_values]
    
    # Create subplots
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Plot function
    axes[0].plot(x_values, y_values, 'b-', linewidth=2, label=f'f(x) = {poly}')
    axes[0].axhline(y=0, color='k', linestyle='-', alpha=0.3)
    axes[0].axvline(x=0, color='k', linestyle='-', alpha=0.3)
    axes[0].set_title('Function')
    axes[0].set_xlabel('x')
    axes[0].set_ylabel('f(x)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Plot first derivative
    axes[1].plot(x_values, y_first, 'r-', linewidth=2, label=f"f'(x) = {first_deriv}")
    axes[1].axhline(y=0, color='k', linestyle='-', alpha=0.3)
    axes[1].axvline(x=0, color='k', linestyle='-', alpha=0.3)
    axes[1].set_title('First Derivative')
    axes[1].set_xlabel('x')
    axes[1].set_ylabel("f'(x)")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # Plot second derivative
    axes[2].plot(x_values, y_second, 'g-', linewidth=2, label=f"f''(x) = {second_deriv}")
    axes[2].axhline(y=0, color='k', linestyle='-', alpha=0.3)
    axes[2].axvline(x=0, color='k', linestyle='-', alpha=0.3)
    axes[2].set_title('Second Derivative')
    axes[2].set_xlabel('x')
    axes[2].set_ylabel("f''(x)")
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

visualize_polynomial_calculus()

### Application: Polynomial Regression

Let's use our Polynomial class for a simple regression task:

In [ ]:
def fit_polynomial_to_data(x_data: np.ndarray, 
                          y_data: np.ndarray, 
                          degree: int) -> Polynomial:
    """
    Fit a polynomial to data using least squares.
    
    This demonstrates how mathematical objects connect to ML.
    
    Args:
        x_data: Input data points
        y_data: Output data points
        degree: Degree of polynomial to fit
        
    Returns:
        Fitted Polynomial object
    """
    # Use numpy's polyfit (returns coefficients in descending order)
    coeffs_desc = np.polyfit(x_data, y_data, degree)
    
    # Reverse for our ascending order convention
    coeffs_asc = list(reversed(coeffs_desc))
    
    return Polynomial(coeffs_asc)

# Generate noisy data from a cubic function
np.random.seed(42)
x_data = np.linspace(-2, 2, 30)
true_function = lambda x: 0.5 * x**3 - x**2 + 0.5
y_data = true_function(x_data) + np.random.normal(0, 0.3, len(x_data))

# Fit polynomials of different degrees
degrees = [1, 3, 9]
x_smooth = np.linspace(-2, 2, 200)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, degree in enumerate(degrees):
    fitted_poly = fit_polynomial_to_data(x_data, y_data, degree)
    y_smooth = [fitted_poly(x) for x in x_smooth]
    
    axes[idx].scatter(x_data, y_data, alpha=0.5, s=30, label='Data')
    axes[idx].plot(x_smooth, y_smooth, 'r-', linewidth=2, 
                  label=f'Degree {degree} fit')
    axes[idx].set_title(f'Polynomial Degree {degree}')
    axes[idx].set_xlabel('x')
    axes[idx].set_ylabel('y')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Notice: Higher degrees can overfit, capturing noise rather than signal!")

---

## 3. Vectors: Geometry and Linear Algebra <a id='vectors'></a>

Vectors represent quantities with magnitude and direction. In ML, they represent data points, features, or parameters.

In [ ]:
class Vector:
    """A vector in n-dimensional space.
    
    Represents both geometric vectors and data points.
    """
    
    def __init__(self, components: List[float]) -> None:
        """
        Initialize a vector with given components.
        
        Args:
            components: List of vector components [x, y, z, ...]
        """
        if not components:
            raise ValueError("Vector must have at least one component")
        self.components = list(components)
    
    @property
    def dimension(self) -> int:
        """Return the dimension of the vector."""
        return len(self.components)
    
    @property
    def magnitude(self) -> float:
        """Calculate the Euclidean norm (length) of the vector."""
        return sqrt(sum(x**2 for x in self.components))
    
    def normalize(self) -> 'Vector':
        """
        Return a unit vector in the same direction.
        
        Returns:
            A new vector with magnitude 1
        """
        mag = self.magnitude
        if mag == 0:
            raise ValueError("Cannot normalize zero vector")
        return Vector([x / mag for x in self.components])
    
    def __add__(self, other: 'Vector') -> 'Vector':
        """Add two vectors component-wise."""
        if not isinstance(other, Vector):
            return NotImplemented
        
        if self.dimension != other.dimension:
            raise ValueError("Vectors must have same dimension")
        
        new_components = [
            x + y for x, y in zip(self.components, other.components)
        ]
        return Vector(new_components)
    
    def __sub__(self, other: 'Vector') -> 'Vector':
        """Subtract two vectors component-wise."""
        if not isinstance(other, Vector):
            return NotImplemented
        
        if self.dimension != other.dimension:
            raise ValueError("Vectors must have same dimension")
        
        new_components = [
            x - y for x, y in zip(self.components, other.components)
        ]
        return Vector(new_components)
    
    def __mul__(self, scalar: float) -> 'Vector':
        """Multiply vector by a scalar."""
        if not isinstance(scalar, (int, float)):
            return NotImplemented
        
        new_components = [scalar * x for x in self.components]
        return Vector(new_components)
    
    def __rmul__(self, scalar: float) -> 'Vector':
        """Support scalar * vector."""
        return self.__mul__(scalar)
    
    def dot(self, other: 'Vector') -> float:
        """
        Calculate the dot product with another vector.
        
        The dot product measures how much two vectors align:
        - Positive: vectors point in similar directions
        - Zero: vectors are perpendicular
        - Negative: vectors point in opposite directions
        
        Args:
            other: Another vector
            
        Returns:
            The dot product (scalar)
        """
        if self.dimension != other.dimension:
            raise ValueError("Vectors must have same dimension")
        
        return sum(x * y for x, y in zip(self.components, other.components))
    
    def angle_with(self, other: 'Vector') -> float:
        """
        Calculate the angle between two vectors in radians.
        
        Uses the formula: cos(θ) = (u · v) / (||u|| ||v||)
        
        Args:
            other: Another vector
            
        Returns:
            Angle in radians [0, π]
        """
        if self.dimension != other.dimension:
            raise ValueError("Vectors must have same dimension")
        
        dot_product = self.dot(other)
        magnitude_product = self.magnitude * other.magnitude
        
        if magnitude_product == 0:
            raise ValueError("Cannot compute angle with zero vector")
        
        # Clamp to [-1, 1] to handle numerical errors
        cos_angle = max(-1, min(1, dot_product / magnitude_product))
        
        return acos(cos_angle)
    
    def project_onto(self, other: 'Vector') -> 'Vector':
        """
        Project this vector onto another vector.
        
        The projection is the component of this vector in the direction of other.
        Formula: proj_v(u) = ((u · v) / (v · v)) * v
        
        Args:
            other: Vector to project onto
            
        Returns:
            The projection vector
        """
        if self.dimension != other.dimension:
            raise ValueError("Vectors must have same dimension")
        
        other_dot_other = other.dot(other)
        if other_dot_other == 0:
            raise ValueError("Cannot project onto zero vector")
        
        scalar = self.dot(other) / other_dot_other
        return scalar * other
    
    def cross(self, other: 'Vector') -> 'Vector':
        """
        Calculate the cross product (only for 3D vectors).
        
        The cross product produces a vector perpendicular to both inputs.
        Its magnitude equals the area of the parallelogram formed by the vectors.
        
        Args:
            other: Another 3D vector
            
        Returns:
            A vector perpendicular to both inputs
        """
        if self.dimension != 3 or other.dimension != 3:
            raise ValueError("Cross product only defined for 3D vectors")
        
        x1, y1, z1 = self.components
        x2, y2, z2 = other.components
        
        return Vector([
            y1 * z2 - z1 * y2,
            z1 * x2 - x1 * z2,
            x1 * y2 - y1 * x2
        ])
    
    def __repr__(self) -> str:
        """Create a readable string representation."""
        components_str = ', '.join(f"{x:.3f}" for x in self.components)
        return f"Vector([{components_str}])"
    
    @classmethod
    def zero(cls, dimension: int) -> 'Vector':
        """Create a zero vector of given dimension."""
        return cls([0] * dimension)
    
    @classmethod
    def basis(cls, dimension: int, index: int) -> 'Vector':
        """
        Create a standard basis vector.
        
        Args:
            dimension: Dimension of the space
            index: Which component is 1 (0-indexed)
            
        Returns:
            A basis vector (0, ..., 0, 1, 0, ..., 0)
        """
        if index < 0 or index >= dimension:
            raise ValueError(f"Index must be between 0 and {dimension-1}")
        
        components = [0] * dimension
        components[index] = 1
        return cls(components)

### Exploring Vector Operations

In [ ]:
# Create vectors
vec_u = Vector([3, 4])
vec_v = Vector([1, 2])

print(f"u = {vec_u}")
print(f"v = {vec_v}")
print(f"\nMagnitude of u: {vec_u.magnitude:.3f}")
print(f"Dimension: {vec_u.dimension}")

# Vector operations
print(f"\nu + v = {vec_u + vec_v}")
print(f"u - v = {vec_u - vec_v}")
print(f"2 * u = {2 * vec_u}")

# Geometric operations
print(f"\nu · v = {vec_u.dot(vec_v):.3f}")
print(f"Angle between u and v: {vec_u.angle_with(vec_v) * 180 / pi:.2f}°")
print(f"Projection of u onto v: {vec_u.project_onto(vec_v)}")

# 3D cross product
vec_3d_a = Vector([1, 0, 0])
vec_3d_b = Vector([0, 1, 0])
print(f"\n{vec_3d_a} × {vec_3d_b} = {vec_3d_a.cross(vec_3d_b)}")

### Visualizing Vector Operations in 2D

In [ ]:
def visualize_vector_operations() -> None:
    """Visualize vector addition and projection."""
    
    # Create vectors
    vec_u = Vector([3, 2])
    vec_v = Vector([1, 3])
    vec_sum = vec_u + vec_v
    vec_proj = vec_u.project_onto(vec_v)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Plot 1: Vector addition
    ax1.quiver(0, 0, vec_u.components[0], vec_u.components[1], 
              angles='xy', scale_units='xy', scale=1, color='blue', 
              width=0.006, label='u')
    ax1.quiver(0, 0, vec_v.components[0], vec_v.components[1], 
              angles='xy', scale_units='xy', scale=1, color='red', 
              width=0.006, label='v')
    ax1.quiver(0, 0, vec_sum.components[0], vec_sum.components[1], 
              angles='xy', scale_units='xy', scale=1, color='green', 
              width=0.006, label='u + v')
    
    # Parallelogram
    ax1.plot([vec_u.components[0], vec_sum.components[0]], 
            [vec_u.components[1], vec_sum.components[1]], 
            'k--', alpha=0.3)
    ax1.plot([vec_v.components[0], vec_sum.components[0]], 
            [vec_v.components[1], vec_sum.components[1]], 
            'k--', alpha=0.3)
    
    ax1.set_xlim(-1, 5)
    ax1.set_ylim(-1, 6)
    ax1.set_aspect('equal')
    ax1.grid(True, alpha=0.3)
    ax1.set_title('Vector Addition')
    ax1.legend()
    
    # Plot 2: Vector projection
    ax2.quiver(0, 0, vec_u.components[0], vec_u.components[1], 
              angles='xy', scale_units='xy', scale=1, color='blue', 
              width=0.006, label='u')
    ax2.quiver(0, 0, vec_v.components[0], vec_v.components[1], 
              angles='xy', scale_units='xy', scale=1, color='red', 
              width=0.006, label='v')
    ax2.quiver(0, 0, vec_proj.components[0], vec_proj.components[1], 
              angles='xy', scale_units='xy', scale=1, color='purple', 
              width=0.006, label='proj_v(u)')
    
    # Perpendicular line
    ax2.plot([vec_u.components[0], vec_proj.components[0]], 
            [vec_u.components[1], vec_proj.components[1]], 
            'k--', alpha=0.5)
    
    ax2.set_xlim(-1, 4)
    ax2.set_ylim(-1, 4)
    ax2.set_aspect('equal')
    ax2.grid(True, alpha=0.3)
    ax2.set_title('Vector Projection')
    ax2.legend()
    
    plt.tight_layout()
    plt.show()

visualize_vector_operations()

---

## 4. Matrices: Linear Transformations <a id='matrices'></a>

Matrices represent linear transformations and systems of equations. They're fundamental to neural networks and data processing.

In [ ]:
class Matrix:
    """A matrix representing a linear transformation.
    
    Stored as a list of rows for intuitive indexing.
    """
    
    def __init__(self, elements: List[List[float]]) -> None:
        """
        Initialize a matrix from a list of rows.
        
        Args:
            elements: List of rows [[row1], [row2], ...]
        """
        if not elements or not elements[0]:
            raise ValueError("Matrix must have at least one element")
        
        # Verify all rows have same length
        num_cols = len(elements[0])
        if not all(len(row) == num_cols for row in elements):
            raise ValueError("All rows must have same length")
        
        self.elements = [list(row) for row in elements]
    
    @property
    def shape(self) -> Tuple[int, int]:
        """Return the shape (rows, cols) of the matrix."""
        return (len(self.elements), len(self.elements[0]))
    
    @property
    def num_rows(self) -> int:
        """Number of rows."""
        return self.shape[0]
    
    @property
    def num_cols(self) -> int:
        """Number of columns."""
        return self.shape[1]
    
    def __getitem__(self, indices: Tuple[int, int]) -> float:
        """Access element at (row, col)."""
        row, col = indices
        return self.elements[row][col]
    
    def __add__(self, other: 'Matrix') -> 'Matrix':
        """Add two matrices element-wise."""
        if not isinstance(other, Matrix):
            return NotImplemented
        
        if self.shape != other.shape:
            raise ValueError("Matrices must have same shape")
        
        new_elements = [
            [self[i, j] + other[i, j] 
             for j in range(self.num_cols)]
            for i in range(self.num_rows)
        ]
        
        return Matrix(new_elements)
    
    def __mul__(self, other: Union['Matrix', Vector, float]) -> Union['Matrix', Vector]:
        """
        Matrix multiplication.
        
        Supports:
        - Matrix @ Matrix -> Matrix
        - Matrix @ Vector -> Vector
        - Matrix @ scalar -> Matrix
        """
        if isinstance(other, Matrix):
            # Matrix multiplication
            if self.num_cols != other.num_rows:
                raise ValueError(
                    f"Cannot multiply {self.shape} by {other.shape}"
                )
            
            new_elements = [
                [
                    sum(self[i, k] * other[k, j] 
                        for k in range(self.num_cols))
                    for j in range(other.num_cols)
                ]
                for i in range(self.num_rows)
            ]
            
            return Matrix(new_elements)
        
        elif isinstance(other, Vector):
            # Matrix-vector multiplication
            if self.num_cols != other.dimension:
                raise ValueError(
                    f"Cannot multiply {self.shape} matrix by "
                    f"{other.dimension}D vector"
                )
            
            new_components = [
                sum(self[i, j] * other.components[j] 
                    for j in range(self.num_cols))
                for i in range(self.num_rows)
            ]
            
            return Vector(new_components)
        
        elif isinstance(other, (int, float)):
            # Scalar multiplication
            new_elements = [
                [self[i, j] * other for j in range(self.num_cols)]
                for i in range(self.num_rows)
            ]
            return Matrix(new_elements)
        
        return NotImplemented
    
    def __rmul__(self, scalar: float) -> 'Matrix':
        """Support scalar * matrix."""
        return self.__mul__(scalar)
    
    def transpose(self) -> 'Matrix':
        """
        Return the transpose of the matrix.
        
        The transpose swaps rows and columns.
        """
        new_elements = [
            [self[i, j] for i in range(self.num_rows)]
            for j in range(self.num_cols)
        ]
        return Matrix(new_elements)
    
    def trace(self) -> float:
        """
        Calculate the trace (sum of diagonal elements).
        
        Only defined for square matrices.
        """
        if self.num_rows != self.num_cols:
            raise ValueError("Trace only defined for square matrices")
        
        return sum(self[i, i] for i in range(self.num_rows))
    
    def determinant(self) -> float:
        """
        Calculate the determinant (only for small matrices).
        
        Implemented for 2x2 and 3x3 matrices.
        """
        if self.num_rows != self.num_cols:
            raise ValueError("Determinant only defined for square matrices")
        
        if self.num_rows == 1:
            return self[0, 0]
        
        elif self.num_rows == 2:
            return self[0, 0] * self[1, 1] - self[0, 1] * self[1, 0]
        
        elif self.num_rows == 3:
            # Sarrus rule for 3x3
            positive = (
                self[0, 0] * self[1, 1] * self[2, 2] +
                self[0, 1] * self[1, 2] * self[2, 0] +
                self[0, 2] * self[1, 0] * self[2, 1]
            )
            negative = (
                self[0, 2] * self[1, 1] * self[2, 0] +
                self[0, 0] * self[1, 2] * self[2, 1] +
                self[0, 1] * self[1, 0] * self[2, 2]
            )
            return positive - negative
        
        else:
            raise NotImplementedError(
                "Determinant only implemented for matrices up to 3x3"
            )
    
    def __repr__(self) -> str:
        """Create a readable string representation."""
        rows_str = []
        for row in self.elements:
            row_str = '[' + ', '.join(f"{x:7.3f}" for x in row) + ']'
            rows_str.append(row_str)
        
        return 'Matrix([\n  ' + ',\n  '.join(rows_str) + '\n])'
    
    @classmethod
    def identity(cls, size: int) -> 'Matrix':
        """
        Create an identity matrix of given size.
        
        The identity matrix leaves vectors unchanged under multiplication.
        """
        elements = [
            [1.0 if i == j else 0.0 for j in range(size)]
            for i in range(size)
        ]
        return cls(elements)
    
    @classmethod
    def rotation_2d(cls, angle: float) -> 'Matrix':
        """
        Create a 2D rotation matrix.
        
        Args:
            angle: Rotation angle in radians
            
        Returns:
            A 2x2 rotation matrix
        """
        cos_a = np.cos(angle)
        sin_a = np.sin(angle)
        
        return cls([
            [cos_a, -sin_a],
            [sin_a, cos_a]
        ])

### Exploring Matrix Operations

In [ ]:
# Create matrices
mat_a = Matrix([[1, 2], [3, 4]])
mat_b = Matrix([[5, 6], [7, 8]])
vec = Vector([1, 2])

print("Matrix A:")
print(mat_a)
print(f"\nShape: {mat_a.shape}")
print(f"Trace: {mat_a.trace()}")
print(f"Determinant: {mat_a.determinant()}")

print("\nMatrix operations:")
print(f"A + B:")
print(mat_a + mat_b)

print(f"\nA * B (matrix multiplication):")
print(mat_a * mat_b)

print(f"\nA * v (matrix-vector product):")
print(mat_a * vec)

print(f"\nTranspose of A:")
print(mat_a.transpose())

### Visualizing Matrix Transformations

In [ ]:
def visualize_matrix_transformation() -> None:
    """Visualize how matrices transform a set of vectors."""
    
    # Create a grid of points
    num_points = 10
    grid_points = [
        Vector([x, y])
        for x in np.linspace(-1, 1, num_points)
        for y in np.linspace(-1, 1, num_points)
    ]
    
    # Create different transformation matrices
    transformations = {
        'Rotation (45°)': Matrix.rotation_2d(pi / 4),
        'Scaling': Matrix([[2, 0], [0, 0.5]]),
        'Shear': Matrix([[1, 0.5], [0, 1]])
    }
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    for idx, (name, matrix) in enumerate(transformations.items()):
        # Apply transformation
        transformed = [matrix * point for point in grid_points]
        
        # Extract coordinates
        original_x = [p.components[0] for p in grid_points]
        original_y = [p.components[1] for p in grid_points]
        trans_x = [p.components[0] for p in transformed]
        trans_y = [p.components[1] for p in transformed]
        
        # Plot
        axes[idx].scatter(original_x, original_y, alpha=0.3, s=30, 
                         c='blue', label='Original')
        axes[idx].scatter(trans_x, trans_y, alpha=0.6, s=30, 
                         c='red', label='Transformed')
        
        # Add basis vectors
        basis_i = Vector([1, 0])
        basis_j = Vector([0, 1])
        trans_i = matrix * basis_i
        trans_j = matrix * basis_j
        
        axes[idx].quiver(0, 0, trans_i.components[0], trans_i.components[1],
                        angles='xy', scale_units='xy', scale=1, 
                        color='darkred', width=0.008)
        axes[idx].quiver(0, 0, trans_j.components[0], trans_j.components[1],
                        angles='xy', scale_units='xy', scale=1, 
                        color='darkred', width=0.008)
        
        axes[idx].set_xlim(-2.5, 2.5)
        axes[idx].set_ylim(-2.5, 2.5)
        axes[idx].set_aspect('equal')
        axes[idx].grid(True, alpha=0.3)
        axes[idx].set_title(name)
        axes[idx].legend()
    
    plt.tight_layout()
    plt.show()

visualize_matrix_transformation()

---

## 5. Composition: Building Complex Mathematical Systems <a id='composition'></a>

Mathematical objects can be composed to create more complex structures. Let's see how our classes work together.

In [ ]:
def demonstrate_composition() -> None:
    """Show how mathematical objects compose."""
    
    # Example: Parametric curve defined by polynomials
    # x(t) = t, y(t) = t^2 - 1
    x_poly = Polynomial([0, 1])      # t
    y_poly = Polynomial([-1, 0, 1])  # t^2 - 1
    
    # Generate curve points
    t_values = np.linspace(-2, 2, 100)
    curve_points = [
        Vector([x_poly(t), y_poly(t)]) 
        for t in t_values
    ]
    
    # Apply a transformation (rotation)
    rotation = Matrix.rotation_2d(pi / 6)
    transformed_points = [rotation * point for point in curve_points]
    
    # Visualize
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Original curve
    orig_x = [p.components[0] for p in curve_points]
    orig_y = [p.components[1] for p in curve_points]
    ax1.plot(orig_x, orig_y, 'b-', linewidth=2, label='Original')
    ax1.set_aspect('equal')
    ax1.grid(True, alpha=0.3)
    ax1.set_title(f'Parametric Curve\nx(t) = {x_poly}, y(t) = {y_poly}')
    ax1.legend()
    
    # Transformed curve
    trans_x = [p.components[0] for p in transformed_points]
    trans_y = [p.components[1] for p in transformed_points]
    ax2.plot(orig_x, orig_y, 'b--', linewidth=1, alpha=0.3, label='Original')
    ax2.plot(trans_x, trans_y, 'r-', linewidth=2, label='Rotated 30°')
    ax2.set_aspect('equal')
    ax2.grid(True, alpha=0.3)
    ax2.set_title('After Rotation Transformation')
    ax2.legend()
    
    plt.tight_layout()
    plt.show()
    
    # Tangent vector using derivative
    t_point = 1.0
    dx_dt = x_poly.derivative()
    dy_dt = y_poly.derivative()
    tangent = Vector([dx_dt(t_point), dy_dt(t_point)])
    
    print(f"\nAt t = {t_point}:")
    print(f"Position: {Vector([x_poly(t_point), y_poly(t_point)])}")
    print(f"Tangent vector: {tangent}")
    print(f"Speed (tangent magnitude): {tangent.magnitude:.3f}")

demonstrate_composition()

---

## 6. Applications in Machine Learning <a id='ml-applications'></a>

Let's see how these mathematical objects relate to machine learning concepts.

In [ ]:
def simple_linear_regression_with_classes() -> None:
    """
    Implement linear regression using our Vector and Matrix classes.
    
    This shows how mathematical abstractions map to ML algorithms.
    """
    
    # Generate synthetic data: y = 2x + 1 + noise
    np.random.seed(42)
    num_samples = 50
    x_data = np.random.uniform(0, 10, num_samples)
    y_data = 2 * x_data + 1 + np.random.normal(0, 1, num_samples)
    
    # Convert to our Vector class
    data_vectors = [
        Vector([1, x])  # Add bias term
        for x in x_data
    ]
    target_vector = Vector(list(y_data))
    
    # Build design matrix X (each row is [1, x_i])
    design_matrix = Matrix([[1, x] for x in x_data])
    
    # Normal equation: θ = (X^T X)^(-1) X^T y
    # For simplicity, we'll use numpy for the inverse
    X = np.array([[1, x] for x in x_data])
    y = y_data.reshape(-1, 1)
    
    # Compute (X^T X)^(-1) X^T y
    XtX = X.T @ X
    XtX_inv = np.linalg.inv(XtX)
    Xty = X.T @ y
    theta = XtX_inv @ Xty
    
    # Create the learned linear function as a polynomial
    learned_function = Polynomial([theta[0, 0], theta[1, 0]])
    
    print(f"Learned function: y = {learned_function}")
    print(f"True function: y = 1.0 + 2.0*x")
    
    # Visualize
    x_plot = np.linspace(0, 10, 100)
    y_pred = [learned_function(x) for x in x_plot]
    
    plt.figure(figsize=(10, 6))
    plt.scatter(x_data, y_data, alpha=0.5, s=50, label='Data')
    plt.plot(x_plot, y_pred, 'r-', linewidth=2, label='Learned function')
    plt.plot(x_plot, 2*x_plot + 1, 'g--', linewidth=2, label='True function', alpha=0.7)
    plt.xlabel('x')
    plt.ylabel('y')
    plt.title('Linear Regression Using Mathematical Objects')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    
    # Calculate residuals using vectors
    predictions = [learned_function(x) for x in x_data]
    residual_vector = Vector([y_data[i] - predictions[i] 
                             for i in range(num_samples)])
    
    print(f"\nResidual norm: {residual_vector.magnitude:.3f}")
    print(f"Mean squared error: {residual_vector.magnitude**2 / num_samples:.3f}")

simple_linear_regression_with_classes()

### Gradient Descent Using Vector Operations

In [ ]:
def gradient_descent_with_vectors() -> None:
    """
    Implement gradient descent using our Vector class.
    
    This shows how optimization algorithms work with mathematical objects.
    """
    
    # Define a simple quadratic function: f(x, y) = x^2 + y^2
    def objective(params: Vector) -> float:
        """Function to minimize."""
        return sum(x**2 for x in params.components)
    
    def gradient(params: Vector) -> Vector:
        """Gradient of the objective."""
        return Vector([2 * x for x in params.components])
    
    # Initialize parameters
    current_params = Vector([3.0, 2.0])
    learning_rate = 0.1
    num_iterations = 50
    
    # Track optimization path
    path = [current_params]
    values = [objective(current_params)]
    
    # Gradient descent
    for iteration in range(num_iterations):
        grad = gradient(current_params)
        current_params = current_params - learning_rate * grad
        path.append(current_params)
        values.append(objective(current_params))
    
    print(f"Initial parameters: {path[0]}")
    print(f"Final parameters: {path[-1]}")
    print(f"Initial value: {values[0]:.6f}")
    print(f"Final value: {values[-1]:.6f}")
    
    # Visualize optimization path
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Contour plot with path
    x_range = np.linspace(-4, 4, 100)
    y_range = np.linspace(-4, 4, 100)
    X, Y = np.meshgrid(x_range, y_range)
    Z = X**2 + Y**2
    
    ax1.contour(X, Y, Z, levels=20, cmap='viridis', alpha=0.6)
    path_x = [p.components[0] for p in path]
    path_y = [p.components[1] for p in path]
    ax1.plot(path_x, path_y, 'ro-', markersize=4, linewidth=2, label='Optimization path')
    ax1.plot(path_x[0], path_y[0], 'go', markersize=10, label='Start')
    ax1.plot(path_x[-1], path_y[-1], 'r*', markersize=15, label='End')
    ax1.set_xlabel('x')
    ax1.set_ylabel('y')
    ax1.set_title('Gradient Descent Path')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_aspect('equal')
    
    # Objective value over iterations
    ax2.plot(values, 'b-', linewidth=2)
    ax2.set_xlabel('Iteration')
    ax2.set_ylabel('Objective Value')
    ax2.set_title('Convergence')
    ax2.grid(True, alpha=0.3)
    ax2.set_yscale('log')
    
    plt.tight_layout()
    plt.show()

gradient_descent_with_vectors()

---

## 7. Summary and Key Principles <a id='summary'></a>

### What We've Built

We've created mathematical objects that:
- **Polynomials**: Support algebra and calculus operations
- **Vectors**: Enable geometric reasoning and linear algebra
- **Matrices**: Represent linear transformations
- **Compose naturally**: Work together to solve complex problems

### Key principles from our implementations:

1. **Immutability**: Operations return new objects (functional style)
2. **Type Checking**: Validate inputs and use `NotImplemented`
3. **Alternate Constructors**: Class methods for different creation patterns
4. **Composition**: Build complex objects from simpler ones
5. **Clear Interfaces**: Methods have single, well-defined purposes

### Applications in Machine Learning

These mathematical objects underpin ML algorithms:

- **Polynomials**: Feature engineering, regression
- **Vectors**: Data points, embeddings, gradients
- **Matrices**: Weight matrices, transformations
- **Functions**: Loss functions, activations, objectives

### From Mathematics to Computation

The progression from mathematical notation to code:

1. **Mathematical concept**: Abstract definition
2. **Data representation**: Choose internal structure
3. **Operations**: Implement as methods
4. **Special methods**: Make syntax natural
5. **Validation**: Ensure mathematical consistency

---

## Next Steps

In **Tutorial F: Building Neural Network Components from Scratch**, we'll apply everything we've learned to create a complete neural network framework:

- Layer classes (Dense, Activation)
- Loss function classes
- Optimizer classes (SGD, Adam)
- Network class to compose everything
- Training loop implementation

This will show how OOP principles enable us to build sophisticated ML systems that are modular, testable, and maintainable - just like professional frameworks such as PyTorch and TensorFlow.